# Neural Prefetcher Zoo: MLP, Perceptron, CNN, RNN/LSTM, Transformer

**One notebook, five models, one comparison table.** Each model is trained on the same memory trace using the same input features, then compared on accuracy and parameter count. The pipeline outputs a ChampSim-compatible `(instr_id, prefetch_address)` text file you can feed into `Quangmire/ChampSim`.

**Why offline?** The papers (Voyager, DART, Twilight) all train offline and either deploy the model or distill it into a hardware-friendly table. We follow that pipeline.

## Outline
1. Setup + download a trace
2. Build the dataset (PC + delta history → next-offset)
3. Define 5 models (Perceptron, MLP, CNN, LSTM, Transformer)
4. Train all 5, log accuracy + parameter count
5. Generate ChampSim prefetch list (using best model)
6. Plot the comparison table

## 1. Setup

In [ ]:
import os, math, random, json, time, io, gzip, urllib.request
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print('Using device:', DEVICE)

### 1b. Get a memory trace

Two paths:
- **Real ML-prefetching trace** from the Quangmire/MLArchSys competition (preferred). Each line is `instr_id, cycle, address, PC, hit/miss`.
- **Synthetic fallback trace** that we generate in-memory (so the notebook runs everywhere). The fallback mixes streaming + pointer-chase patterns, which is enough to compare models for the slide demo.

If the real download fails (e.g., the Box mirror is rate-limited from Colab), set `USE_SYNTHETIC = True`.

In [ ]:
USE_SYNTHETIC = True   # set False if you have a real .txt trace URL handy
TRACE_URL = ''         # e.g. a raw .txt file (instr_id,cycle,addr,PC,hit/miss)
MAX_LINES = 200_000    # cap for Colab memory

def load_real_trace(url, max_lines):
    print('Downloading', url)
    raw = urllib.request.urlopen(url).read()
    if url.endswith('.gz'):
        raw = gzip.decompress(raw)
    text = raw.decode('utf-8', errors='ignore').splitlines()[:max_lines]
    rows = []
    for ln in text:
        parts = ln.strip().split(',')
        if len(parts) < 4: continue
        try:
            iid = int(parts[0])
            addr = int(parts[2], 0)
            pc   = int(parts[3], 0)
            rows.append((iid, addr, pc))
        except ValueError:
            continue
    return rows

def make_synthetic_trace(n=200_000, seed=0):
    """Mix of learnable patterns -- chosen so the prefetcher zoo can
    actually demonstrate accuracy differences instead of random output.

    (a) Streaming PC : linear stride +1 cache-line (deterministic).
    (b) Stride-2 PC  : linear stride +2 cache-lines (deterministic).
    (c) Stride-8 PC  : linear stride +8 cache-lines (deterministic).
    (d) Loop-back PC : sweep [0..15] cache-lines and restart (loop).
    (e) Markov PC    : 4-state Markov chain over a small set of strides.
    (f) Noise PC     : random offset (the unlearnable baseline).

    Each pattern has its own PC tag, so a model that uses PC well will
    quickly outperform a model that doesn't. Most accesses are within
    the same page so next-offset classification is well-defined.
    """
    rng = np.random.default_rng(seed)
    PCS = {'stream':0xA000,'stride2':0xA100,'stride8':0xA200,
           'loop':0xA300,'markov':0xA400,'noise':0xA500}
    PAGE = 0x1_0000_0000
    rows = []; iid = 0
    # Per-PC running pointers (within a single page so labels stay in [0,127])
    ptr = {k: 0 for k in PCS}
    markov_states = [1, 2, 4, 8]              # 4 stride choices
    markov_trans  = {0:[0.7,0.2,0.05,0.05],   # mostly +1
                     1:[0.1,0.7,0.15,0.05],   # mostly +2
                     2:[0.05,0.15,0.7,0.10],  # mostly +4
                     3:[0.05,0.10,0.15,0.7]}  # mostly +8
    markov_s = 0
    pattern_keys = list(PCS.keys())
    while len(rows) < n:
        # Pick a pattern (uniform). 6 PCs total.
        key = pattern_keys[rng.integers(0, len(pattern_keys))]
        for _ in range(rng.integers(8, 25)):     # short burst per pattern
            if   key == 'stream':  ptr[key] = (ptr[key] + 1)  % 128
            elif key == 'stride2': ptr[key] = (ptr[key] + 2)  % 128
            elif key == 'stride8': ptr[key] = (ptr[key] + 8)  % 128
            elif key == 'loop':    ptr[key] = (ptr[key] + 1)  % 16
            elif key == 'markov':
                probs = markov_trans[markov_s]
                markov_s = int(rng.choice(4, p=probs))
                ptr[key] = (ptr[key] + markov_states[markov_s]) % 128
            else:                                # noise
                ptr[key] = int(rng.integers(0, 128))
            addr = PAGE | (ptr[key] << 6)        # 64-byte line within the page
            rows.append((iid, int(addr), PCS[key])); iid += 1
            if len(rows) >= n: break
    return rows[:n]

if USE_SYNTHETIC or not TRACE_URL:
    rows = make_synthetic_trace(n=MAX_LINES)
    print('Using synthetic trace, lines =', len(rows))
else:
    rows = load_real_trace(TRACE_URL, MAX_LINES)
    print('Loaded real trace, lines =', len(rows))
print('Example row (instr_id, addr, PC):', rows[0])

## 2. Build the dataset

**Input features** (UBC MLP 2021 / Voyager 2021 / DART 2024 all share this base):
- PC hash: 12 bits
- Last 4 deltas (signed, quantized to 8-bit buckets)

**Output**: next cache-line offset within the same 4 KiB page (128-class softmax).
We frame next-line prediction as classification over `[0, 127]` page-offsets. This avoids the 2^64 address-space explosion that Voyager and Hashemi flagged.

In [ ]:
HIST = 4                      # delta history length
NUM_PC_HASH = 4096            # 12-bit hash bucket
DELTA_VOCAB = 257             # 0..255 + special 'large' token
NUM_CLASSES = 128             # next page-offset
PAGE_BITS = 12; LINE_BITS = 6  # 4 KiB pages, 64 B lines  --> 64 offsets/page
                                # we use 128 just in case a delta crosses pages.

def quantize_delta(d):
    """Map signed delta to [0,256]. >0 small => 1..127. <0 small => 128..254. Big => 255."""
    d_lines = d >> LINE_BITS      # in cache-line units
    if d_lines == 0: return 0
    if d_lines > 0 and d_lines <= 127: return d_lines
    if d_lines < 0 and d_lines >= -126: return 128 + (-d_lines)
    return 255

def build_examples(rows):
    X_pc = []; X_d = []; Y = []; meta = []
    addrs_by_pc = {}                 # PC -> deque of last few addresses
    for (iid, addr, pc) in rows:
        prev = addrs_by_pc.get(pc, [])
        # Build delta history relative to *previous* accesses of the same PC
        deltas = []
        for k in range(HIST):
            if len(prev) > k+1:
                deltas.append(quantize_delta(prev[-(k+1)] - prev[-(k+2)]))
            else:
                deltas.append(0)
        # Label = next-line offset within current page
        page = addr >> PAGE_BITS
        next_addr_within_page = (addr >> LINE_BITS) & 0x7F
        if len(prev) >= 1:           # we need a 'prev' to form an example
            X_pc.append(pc & (NUM_PC_HASH - 1))
            X_d.append(deltas)
            Y.append(next_addr_within_page)
            meta.append((iid, page, addr))
        prev.append(addr); prev = prev[-8:]
        addrs_by_pc[pc] = prev
    return (np.array(X_pc, dtype=np.int64),
            np.array(X_d,  dtype=np.int64),
            np.array(Y,    dtype=np.int64),
            meta)

X_pc, X_d, Y, meta = build_examples(rows)
print('Examples:', len(Y), '| feature shapes:', X_pc.shape, X_d.shape)
print('Label dist (first 10 classes):',
      np.bincount(Y, minlength=NUM_CLASSES)[:10])

In [ ]:
class PFDataset(Dataset):
    def __init__(self, pc, d, y):
        self.pc = torch.from_numpy(pc); self.d = torch.from_numpy(d); self.y = torch.from_numpy(y)
    def __len__(self):    return len(self.y)
    def __getitem__(self, i): return self.pc[i], self.d[i], self.y[i]

ds = PFDataset(X_pc, X_d, Y)
n_train = int(0.8 * len(ds))
tr_ds, va_ds = random_split(ds, [n_train, len(ds) - n_train],
                            generator=torch.Generator().manual_seed(0))
BATCH = 512
tr_loader = DataLoader(tr_ds, batch_size=BATCH, shuffle=True,  drop_last=True)
va_loader = DataLoader(va_ds, batch_size=BATCH, shuffle=False, drop_last=False)
print('train/val:', len(tr_ds), len(va_ds))

## 3. Five Model Definitions

Each model has the **same input/output interface** so the training loop is shared.
- Input  : `(pc:int64 [B], d:int64 [B,HIST])`
- Output : logits `[B, NUM_CLASSES]`

In [ ]:
EMB_PC = 32
EMB_D  = 16

class Perceptron(nn.Module):
    """Jiménez-style single-layer perceptron. Input is one-hot-ish features.
    We embed PC and deltas, concatenate, then apply a single linear -> NUM_CLASSES.
    No non-linearity (true to the perceptron definition)."""
    def __init__(self):
        super().__init__()
        self.epc = nn.Embedding(NUM_PC_HASH, EMB_PC)
        self.ed  = nn.Embedding(DELTA_VOCAB, EMB_D)
        self.fc  = nn.Linear(EMB_PC + HIST*EMB_D, NUM_CLASSES)
    def forward(self, pc, d):
        x = torch.cat([self.epc(pc), self.ed(d).flatten(1)], dim=1)
        return self.fc(x)

class MLP(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.epc = nn.Embedding(NUM_PC_HASH, EMB_PC)
        self.ed  = nn.Embedding(DELTA_VOCAB, EMB_D)
        D_IN = EMB_PC + HIST*EMB_D
        self.net = nn.Sequential(
            nn.Linear(D_IN, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, NUM_CLASSES))
    def forward(self, pc, d):
        x = torch.cat([self.epc(pc), self.ed(d).flatten(1)], dim=1)
        return self.net(x)

class CNN1D(nn.Module):
    """1-D CNN over the delta history, conditioned on PC.
    Useful for detecting *repeated stride patterns* across the last HIST deltas."""
    def __init__(self, ch=32):
        super().__init__()
        self.epc = nn.Embedding(NUM_PC_HASH, EMB_PC)
        self.ed  = nn.Embedding(DELTA_VOCAB, EMB_D)
        self.conv = nn.Sequential(
            nn.Conv1d(EMB_D, ch, kernel_size=2, padding=1), nn.ReLU(),
            nn.Conv1d(ch, ch, kernel_size=2, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Linear(ch + EMB_PC, NUM_CLASSES)
    def forward(self, pc, d):
        e = self.ed(d).transpose(1, 2)        # [B, EMB_D, HIST]
        c = self.conv(e).squeeze(-1)          # [B, ch]
        return self.fc(torch.cat([self.epc(pc), c], dim=1))

class LSTMNet(nn.Module):
    """LSTM over delta sequence; matches the Hashemi 2018 family."""
    def __init__(self, hidden=64):
        super().__init__()
        self.epc = nn.Embedding(NUM_PC_HASH, EMB_PC)
        self.ed  = nn.Embedding(DELTA_VOCAB, EMB_D)
        self.lstm = nn.LSTM(EMB_D, hidden, batch_first=True)
        self.fc = nn.Linear(hidden + EMB_PC, NUM_CLASSES)
    def forward(self, pc, d):
        seq = self.ed(d)                                 # [B, HIST, EMB_D]
        out, (h, _) = self.lstm(seq)                     # h: [1,B,hidden]
        return self.fc(torch.cat([self.epc(pc), h[0]], dim=1))

class TinyTransformer(nn.Module):
    """2-layer 4-head Transformer encoder over delta sequence.
    Voyager-lite: attention picks the relevant past delta."""
    def __init__(self, dmodel=32, nhead=4, layers=2):
        super().__init__()
        self.epc = nn.Embedding(NUM_PC_HASH, EMB_PC)
        self.ed  = nn.Embedding(DELTA_VOCAB, dmodel)
        self.pos = nn.Parameter(torch.randn(HIST, dmodel) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=dmodel, nhead=nhead,
                                               dim_feedforward=64,
                                               batch_first=True, dropout=0.0)
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.fc = nn.Linear(dmodel + EMB_PC, NUM_CLASSES)
    def forward(self, pc, d):
        z = self.ed(d) + self.pos                          # [B, HIST, dmodel]
        z = self.enc(z)                                    # [B, HIST, dmodel]
        z = z.mean(dim=1)                                  # pool
        return self.fc(torch.cat([self.epc(pc), z], dim=1))

MODELS = {
    'Perceptron':   Perceptron,
    'MLP':          MLP,
    'CNN':          CNN1D,
    'LSTM':         LSTMNet,
    'Transformer':  TinyTransformer,
}

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

for name, cls in MODELS.items():
    m = cls()
    print(f'{name:14s}  params = {count_params(m):>10,d}')

## 4. Train all five and log accuracy

In [ ]:
EPOCHS = 6
LR = 2e-3

def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for pc, d, y in loader:
            pc, d, y = pc.to(DEVICE), d.to(DEVICE), y.to(DEVICE)
            logits = model(pc, d)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.numel()
    return correct / total

def train_model(name, cls):
    model = cls().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    t0 = time.time()
    best_acc = 0.0
    for ep in range(EPOCHS):
        model.train()
        running = 0.0; nb = 0
        for pc, d, y in tr_loader:
            pc, d, y = pc.to(DEVICE), d.to(DEVICE), y.to(DEVICE)
            logits = model(pc, d)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item(); nb += 1
        sched.step()
        va_acc = evaluate(model, va_loader)
        best_acc = max(best_acc, va_acc)
        print(f'  [{name}] epoch {ep+1}/{EPOCHS}  loss={running/max(1,nb):.4f}  va_acc={va_acc:.4f}')
    t_total = time.time() - t0
    return model, best_acc, t_total

results = {}
for name, cls in MODELS.items():
    print(f'\n=== Training {name} ===')
    m, acc, t = train_model(name, cls)
    results[name] = dict(model=m, va_acc=acc, train_sec=t, params=count_params(m))
    print(f'  -> best va_acc={acc:.4f}  params={results[name]["params"]:,}  train_sec={t:.1f}')

# ----- summary -----
print('\n' + '='*70)
print(f'{"Model":14s} {"Params":>12s}  {"Val Acc":>10s}  {"Train (s)":>10s}')
print('-'*70)
for n,r in results.items():
    print(f'{n:14s} {r["params"]:>12,d}  {r["va_acc"]:>10.4f}  {r["train_sec"]:>10.1f}')
print('='*70)

## 5. Inference-latency micro-benchmark

We measure single-example forward-pass time on CPU. This is **not the actual hardware deployment latency** -- a real RTL implementation would be in nanoseconds, not milliseconds. But the relative ordering across models still matches what you'd expect from FLOP counts: Perceptron ≪ MLP ≪ CNN ≈ LSTM ≪ Transformer.

Once a winner is chosen, the deployment story is **distillation into a lookup table** (DART / Net2Tab), at which point all five collapse to the same ~1-cycle table-lookup latency.

In [ ]:
def bench_latency(model, n_iter=2000):
    model.eval().to('cpu')
    pc = torch.zeros(1, dtype=torch.long)
    d  = torch.zeros(1, HIST, dtype=torch.long)
    with torch.no_grad():
        # warmup
        for _ in range(20): model(pc, d)
        t0 = time.perf_counter()
        for _ in range(n_iter): model(pc, d)
        dt = (time.perf_counter() - t0) / n_iter
    return dt * 1e6   # microseconds

for n, r in results.items():
    us = bench_latency(r['model'])
    r['inf_us'] = us
    print(f'{n:14s} CPU-inference per example: {us:6.1f} us')

## 6. Generate ChampSim prefetch list using the best model

Format expected by Quangmire/ChampSim's `ml_prefetch_sim.py`:
```
instr_id_in_decimal prefetch_addr_in_hex
```
We run the winning model on the full trace, take top-1 next-offset, recompose with the current page, and emit one prefetch per access.

Open the produced `prefetch_list.txt` in ChampSim with:
```
./ml_prefetch_sim.py run path/to/trace --prefetch /path/to/prefetch_list.txt
```

In [ ]:
best_name = max(results, key=lambda k: results[k]['va_acc'])
print('Best model by val acc:', best_name)
best = results[best_name]['model'].to(DEVICE)
best.eval()

out_path = 'prefetch_list.txt'
with torch.no_grad(), open(out_path, 'w') as fh:
    # iterate the same examples in trace order
    bs = 1024
    for i in range(0, len(X_pc), bs):
        pc = torch.from_numpy(X_pc[i:i+bs]).to(DEVICE)
        d  = torch.from_numpy(X_d[i:i+bs]).to(DEVICE)
        logits = best(pc, d)
        pred_offsets = logits.argmax(dim=1).cpu().numpy()
        for j, off in enumerate(pred_offsets):
            iid, page, addr = meta[i+j]
            pf_addr = (page << PAGE_BITS) | (int(off) << LINE_BITS)
            fh.write(f'{iid} 0x{pf_addr:x}\n')
print('Wrote', out_path, '-- lines:', sum(1 for _ in open(out_path)))

## 7. Comparison chart (for slide 4 backup / replacement)

In [ ]:
import matplotlib.pyplot as plt
fig, ax1 = plt.subplots(figsize=(8,4))
names = list(results.keys())
accs  = [results[n]['va_acc'] for n in names]
params = [results[n]['params'] for n in names]
lats   = [results[n]['inf_us'] for n in names]

x = np.arange(len(names))
ax1.bar(x - 0.25, accs, width=0.5, color='steelblue', label='Val accuracy')
ax1.set_ylabel('Validation accuracy')
ax1.set_xticks(x); ax1.set_xticklabels(names)
ax1.set_ylim(0, max(accs)*1.2)
for i,a in enumerate(accs): ax1.text(i-0.25, a+0.01, f'{a:.2f}', ha='center', fontsize=8)

ax2 = ax1.twinx()
ax2.plot(x, lats, color='darkorange', marker='o', label='Inference us (CPU)')
ax2.set_ylabel('Inference us (CPU)')
ax2.set_yscale('log')
ax1.set_title('NN Prefetcher Family: accuracy vs. latency')
fig.tight_layout()
fig.savefig('nn_family_comparison.png', dpi=140)
plt.show()

# Also dump JSON so the slide deck can ingest the numbers.
summary = {n: {k:v for k,v in r.items() if k != 'model'} for n, r in results.items()}
with open('nn_family_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=2)
print('Saved nn_family_comparison.png and nn_family_summary.json')

## 8. Notes you can put in your slide

- All five models use the **same input features** (PC hash + last 4 deltas) and **same output** (128-class softmax over next page offset). This matches Voyager, Hashemi 2018, DART, and the UBC MLP-prefetcher 2021.
- Accuracy on this synthetic trace is **not the same** as IPC. It's a sanity check that the pipeline works. The real IPC number comes from feeding `prefetch_list.txt` into ChampSim.
- Transformer and LSTM are expected to win on accuracy but lose on latency. Perceptron and MLP are expected to be much faster.
- The hardware story is **distillation into a lookup table** (DART, Net2Tab), which collapses the latency gap. That is Phase 2.
- This notebook ran end-to-end on Colab T4 in roughly 5--10 minutes for the synthetic 200k trace. With a real DPC-3 LoadTrace of 20M records, expect ~30--40 min per model.